In [1]:
import pandas as pd
import numpy as np
import random

def generate_logistics_dataset(num_rows=100000):
    print(f"🏭 Generating {num_rows} rows of Logistics & Demographic data...")
    np.random.seed(42)
    
    # 1. Base IDs
    # Assuming donor_ids match your registry (e.g., 1 to 100,000)
    donor_ids = np.arange(1, num_rows + 1)
    
    # Countries (to map logically to your existing data)
    countries = np.random.choice(['MX', 'CN', 'UG', 'JP', 'TR', 'BR', 'US', 'IN', 'DE'], size=num_rows)
    
    # ---------------------------------------------------------
    # J) TRANSPORT & LOGISTICS (The "Last Mile" problem)
    # ---------------------------------------------------------
    
    # Zone ID (Simulating neighborhoods/sectors)
    # Donors in the same Zone share similar logistics
    zone_ids = [f"{c}-Z{np.random.randint(1, 100):03d}" for c in countries]
    
    # Distance: Log-normal distribution (Most live <10km, some live 50km+)
    distances = np.random.lognormal(mean=2.0, sigma=0.8, size=num_rows).round(2)
    
    # Traffic Factor: Varies by country (1.0 = clear, 2.5 = heavy jam)
    traffic_factors = np.random.uniform(1.0, 2.5, size=num_rows)
    
    # Time (Minutes): Distance * Avg Speed (approx 30km/h in city) * Traffic
    travel_times = (distances * (60/30) * traffic_factors).astype(int)
    
    # Transport Capacity (Slots available in the mobile unit/shuttle for this zone)
    # Rural areas have fewer slots, Urban have more
    transport_capacity = np.random.randint(5, 50, size=num_rows)
    
    # Transfer Cost (Estimated in USD)
    # Base fee + per km cost + cold chain surcharge
    transfer_costs = (5 + (distances * 0.5) + np.random.uniform(0, 5, size=num_rows)).round(2)
    
    # Route Feasibility (Is the road passable?)
    # 95% feasibility, 5% blocked (construction, floods, etc.)
    route_feasibility = np.random.choice([0, 1], size=num_rows, p=[0.05, 0.95])
    
    # Cold Chain Constraint (Quality of thermal preservation)
    # 0.0 (Broken Fridge) to 1.0 (Perfect)
    # Longer distances increase risk of cold chain failure
    cold_chain_risk = (distances / 100).clip(0, 0.5) # Higher distance = higher risk
    cold_chain_quality = 1.0 - (cold_chain_risk * np.random.uniform(0, 1, size=num_rows))
    
    # ---------------------------------------------------------
    # K) SENSITIVE VARIABLES (Aggregated by Area)
    # ---------------------------------------------------------
    # We simulate these based on "Zone Quality" so they are correlated
    
    # Deprivation Index (1 = Wealthy, 10 = Deprived)
    # Randomly assigned to zones (simplified here)
    deprivation_index = np.random.randint(1, 11, size=num_rows)
    
    # Education Index (0.0 to 1.0)
    # Negatively correlated with Deprivation
    education_base = 1 - (deprivation_index / 12)
    area_education_index = (education_base + np.random.normal(0, 0.05, size=num_rows)).clip(0, 1)
    
    # Cultural/Language Mix (0.0 = Homogeneous, 1.0 = Highly Diverse)
    area_language_mix = np.random.beta(2, 5, size=num_rows).round(3)
    
    # ---------------------------------------------------------
    # CREATE DATAFRAME
    # ---------------------------------------------------------
    df = pd.DataFrame({
        'donor_id': donor_ids,
        'country_code': countries,
        'zone_id': zone_ids,
        
        # Logistics (Category J)
        'center_distance_km': distances,
        'travel_time_min': travel_times,
        'transport_capacity_slots': transport_capacity,
        'transfer_cost_usd': transfer_costs,
        'route_feasibility_flag': route_feasibility,
        'cold_chain_quality_score': cold_chain_quality.round(3),
        
        # Geo-Demographics (Category K)
        'area_deprivation_index': deprivation_index, # 1-10
        'area_education_index': area_education_index.round(3), # 0-1
        'area_language_mix': area_language_mix # 0-1
    })
    
    # Save
    filename = 'blood_donor_logistics_extended.csv'
    df.to_csv(filename, index=False)
    print(f"✅ Success! Saved {filename}")
    print(df.head())
    return df

# Generate it
df_logistics = generate_logistics_dataset(100000)

🏭 Generating 100000 rows of Logistics & Demographic data...
✅ Success! Saved blood_donor_logistics_extended.csv
   donor_id country_code  zone_id  center_distance_km  travel_time_min  \
0         1           US  US-Z084                4.92               16   
1         2           JP  JP-Z094               11.18               39   
2         3           IN  IN-Z017               14.06               28   
3         4           TR  TR-Z085                5.39               12   
4         5           US  US-Z068               18.24               42   

   transport_capacity_slots  transfer_cost_usd  route_feasibility_flag  \
0                        36               9.17                       1   
1                        46              13.95                       1   
2                        23              16.47                       1   
3                        19               8.85                       1   
4                        17              16.33                       1   